<a href="https://colab.research.google.com/github/KeertanPatel-CS/movie_recommender-svd/blob/main/notebooks/04_what_it_learned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movie Recommendation System
## Milestone 4: Hidden Factors and Personal Recommendations

**Team members:** Kirtan Patel and Rudra Patel  
**Course:** MATH 344L

We explore the patterns learned by Model 2 and use personal movie
ratings to generate recommendations.

We compare the personalized predictions with the baseline and examine
which movies receive the largest positive and negative adjustments.

In [28]:
import os
import sys
import urllib.request
import zipfile
import numpy as np
import pandas as pd

# Download the project's reusable code into this Colab session.
REPO = "movie_recommender-svd"

if not os.path.exists(REPO):
    !git clone -q https://github.com/KeertanPatel-CS/movie_recommender-svd.git

# Import the functions saved in our repository.
sys.path.insert(0, f"{REPO}/src")

from evaluate import to_matrix
from models import baseline_matrix, rank_k

print("Project functions imported successfully.")

Project functions imported successfully.


This cell imports the Python libraries and our reusable project functions.
We use the same baseline and low-rank reconstruction methods as before.

In [29]:
# Download MovieLens if the ratings file is not already available.
if not os.path.exists("ml-100k/u.data"):
    urllib.request.urlretrieve(
        "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
        "ml-100k.zip"
    )

    with zipfile.ZipFile("ml-100k.zip", "r") as zip_ref:
        zip_ref.extractall()

# Load the user ratings.
ratings = pd.read_csv(
    "ml-100k/u.data",
    sep="\t",
    names=["user", "item", "rating", "time"]
)

# Load movie titles in movie-ID order.
movie_info = pd.read_csv(
    "ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1],
    names=["item", "title"]
).sort_values("item")

titles = movie_info["title"].to_numpy()

print("Number of ratings:", len(ratings))
print("Number of movies:", len(titles))
print("Movie 50:", titles[49])

Number of ratings: 100000
Number of movies: 1682
Movie 50: Star Wars (1977)


The ratings table contains one row per observed rating.
The titles array connects movie positions with readable names.

Movie IDs start at 1, but array positions start at 0.
For example, movie 50 is stored at titles[49].

In [30]:
# Fit Model 2 on all available ratings for interpretation.
R = to_matrix(ratings)
known = ~np.isnan(R)

B = baseline_matrix(R)
X = np.where(known, R - B, 0.0)

# Keep these factors separate from later personalized-model factors.
U_all, s_all, Vt_all = np.linalg.svd(
    X, full_matrices=False
)

# Examine movies with at least 50 ratings.
n_ratings = known.sum(axis=0)
popular = n_ratings >= 50

# Use the rank chosen through validation in Milestone 3.
chosen_k = 15

print("Matrix shape:", R.shape)
print("Movies with at least 50 ratings:", popular.sum())
print("Selected rank:", chosen_k)

Matrix shape: (943, 1682)
Movies with at least 50 ratings: 603
Selected rank: 15


In Milestone 3, we held out ratings to evaluate prediction accuracy.
Here, we fit on all available ratings to interpret the model and
generate recommendations. We do not report test accuracy from this fit.

We subtract the baseline before computing SVD. This lets the factors
describe patterns beyond general user and movie effects.

We inspect movies with at least 50 ratings so that our interpretation
is less influenced by movies with very few observations.

In [31]:
def show_factor(f, n=6):
    """Display popular movies at both ends of a hidden factor."""

    # Factor numbering starts at 1; array indexing starts at 0.
    values = Vt_all[f - 1]

    # Sort eligible movies by their score on this factor.
    movie_positions = np.where(popular)[0]
    order = movie_positions[np.argsort(values[movie_positions])]

    print(f"\nFactor {f}: lowest scores")
    for movie in order[:n]:
        print(f"  {values[movie]:+.3f}  {titles[movie]}")

    print(f"Factor {f}: highest scores")
    for movie in order[::-1][:n]:
        print(f"  {values[movie]:+.3f}  {titles[movie]}")


# Inspect all three factors required by Exercise 1.
show_factor(1)
show_factor(2)
show_factor(3)


Factor 1: lowest scores
  -0.087  Clockwork Orange, A (1971)
  -0.079  Trainspotting (1996)
  -0.072  Leaving Las Vegas (1995)
  -0.072  Pulp Fiction (1994)
  -0.069  Brazil (1985)
  -0.067  Fargo (1996)
Factor 1: highest scores
  +0.211  Independence Day (ID4) (1996)
  +0.175  Return of the Jedi (1983)
  +0.149  Raiders of the Lost Ark (1981)
  +0.149  Back to the Future (1985)
  +0.136  Braveheart (1995)
  +0.134  Forrest Gump (1994)

Factor 2: lowest scores
  -0.099  Independence Day (ID4) (1996)
  -0.092  Twister (1996)
  -0.077  Eraser (1996)
  -0.074  Air Force One (1997)
  -0.067  Saint, The (1997)
  -0.066  Phenomenon (1996)
Factor 2: highest scores
  +0.203  Star Wars (1977)
  +0.199  Pulp Fiction (1994)
  +0.173  Godfather, The (1972)
  +0.168  Empire Strikes Back, The (1980)
  +0.151  Fargo (1996)
  +0.146  Return of the Jedi (1983)

Factor 3: lowest scores
  -0.137  Independence Day (ID4) (1996)
  -0.131  Star Trek: First Contact (1996)
  -0.130  Rock, The (1996)
  -0.129 

Each row of Vt_all assigns movies a score on one hidden factor.
We inspect the movies with the lowest and highest scores to look
for a possible interpretation.

A positive score does not mean a movie is better. The two ends
represent contrasting patterns in ratings.

SVD factor signs can flip without changing the reconstructed matrix.
Therefore, the groups of movies matter more than which group has
positive scores. Any names we give the factors are interpretations,
not labels supplied to the model.

## Personal Movie Recommendations

We will enter ratings for 10–15 movies we have seen, add those ratings
as a new user, and compare Model 2's recommendations with the baseline.

First, we search the dataset to find the correct movie IDs.

In [32]:
def find(word):
    """Print the ID and title of movies matching a search word."""

    for position, title in enumerate(titles):
        # Search without distinguishing uppercase and lowercase letters.
        if word.lower() in title.lower():
            print(position + 1, title)


# Change the search text to look for movies you have seen.
find("toy story")

1 Toy Story (1995)


The search prints each matching movie's ID and title.
We use these IDs to connect our personal ratings to the correct
columns of the ratings matrix.

## My Movie Ratings

I rated ten movies on a 10-point scale and divided each rating by 2
to convert it to the model's star scale. I kept half-star ratings
rather than rounding them.

In [33]:
# My personal ratings on a 10-point scale.
ratings_out_of_10 = {
    "Home Alone (1990)": 10,
    "Titanic (1997)": 10,
    "Jumanji (1995)": 10,
    "Jurassic Park (1993)": 8,
    "Toy Story (1995)": 7,
    "Lion King, The (1994)": 6,
    "Aladdin (1992)": 8,
    "Mask, The (1994)": 8,
    "Men in Black (1997)": 6,
    "Forrest Gump (1994)": 5
}

# Find each movie's ID and convert its rating to stars.
my_ratings = {}

for title, rating in ratings_out_of_10.items():
    matches = movie_info.loc[movie_info["title"] == title, "item"]

    # Stop if a title is missing or ambiguous.
    if len(matches) != 1:
        raise ValueError(f"Check this movie title: {title}")

    movie_id = int(matches.iloc[0])
    my_ratings[movie_id] = rating / 2

    print(f"{movie_id:4d} | {rating / 2:.1f} stars | {title}")

  94 | 5.0 stars | Home Alone (1990)
 313 | 5.0 stars | Titanic (1997)
 755 | 5.0 stars | Jumanji (1995)
  82 | 4.0 stars | Jurassic Park (1993)
   1 | 3.5 stars | Toy Story (1995)
  71 | 3.0 stars | Lion King, The (1994)
  95 | 4.0 stars | Aladdin (1992)
  72 | 4.0 stars | Mask, The (1994)
 257 | 3.0 stars | Men in Black (1997)
  69 | 2.5 stars | Forrest Gump (1994)


This code matches each title to its movie ID and stores my star rating
in a dictionary. Some dataset titles place "The" at the end, such as
"Lion King, The."

Matching titles avoids guessing movie IDs and assigning a rating
to the wrong movie.

In [34]:
# Create one new row with missing values for unrated movies.
me = np.full((1, R.shape[1]), np.nan)

# Place my ratings in the correct movie columns.
for movie_id, stars in my_ratings.items():
    me[0, movie_id - 1] = stars

# Add myself below the original 943 users.
R_me = np.vstack([R, me])

# Refit the baseline and residual SVD with my ratings included.
known_me = ~np.isnan(R_me)
B_me = baseline_matrix(R_me)
X_me = np.where(known_me, R_me - B_me, 0.0)

U_me, s_me, Vt_me = np.linalg.svd(
    X_me, full_matrices=False
)

# Use k = 15, selected through validation in Milestone 3.
P_me = rank_k(U_me, s_me, Vt_me, chosen_k) + B_me

print("Matrix shape:", R_me.shape)
print("My number of ratings:", np.count_nonzero(~np.isnan(me)))
print("Rank used:", chosen_k)

Matrix shape: (944, 1682)
My number of ratings: 10
Rank used: 15


My ratings become the last row of a matrix with 944 users.
The remaining entries in my row stay unknown.

We refit Model 2 using the expanded matrix and keep k = 15,
the rank selected through validation in Milestone 3.
The original factor-analysis variables remain unchanged.

In [35]:
# My predictions are in the final row.
mine = P_me[-1]
base = B_me[-1]

# Recommend only movies with at least 50 original ratings
# that I have not already rated.
can_pick = popular & np.isnan(R_me[-1])


def top(scores, n=10):
    """Return positions of the highest-scoring eligible movies."""

    eligible = np.where(can_pick)[0]
    order = np.argsort(-scores[eligible])
    return eligible[order[:n]]


my_top10 = top(mine)
baseline_top10 = top(base)

print("Model 2 recommends:")
for movie in my_top10:
    print(f"  {mine[movie]:.2f} | {titles[movie]}")

print("\nBaseline recommends:")
for movie in baseline_top10:
    print(f"  {base[movie]:.2f} | {titles[movie]}")

overlap = len(set(my_top10) & set(baseline_top10))
print("\nMovies in both top-10 lists:", overlap)

Model 2 recommends:
  4.83 | Close Shave, A (1995)
  4.73 | Wrong Trousers, The (1993)
  4.72 | Usual Suspects, The (1995)
  4.68 | Casablanca (1942)
  4.67 | Wallace & Gromit: The Best of Aardman Animation (1996)
  4.67 | Schindler's List (1993)
  4.66 | Shawshank Redemption, The (1994)
  4.63 | Good Will Hunting (1997)
  4.60 | 12 Angry Men (1957)
  4.60 | As Good As It Gets (1997)

Baseline recommends:
  4.83 | Close Shave, A (1995)
  4.75 | Wrong Trousers, The (1993)
  4.73 | Schindler's List (1993)
  4.73 | Shawshank Redemption, The (1994)
  4.69 | Casablanca (1942)
  4.68 | Wallace & Gromit: The Best of Aardman Animation (1996)
  4.68 | Usual Suspects, The (1995)
  4.66 | Good Will Hunting (1997)
  4.65 | Star Wars (1977)
  4.61 | 12 Angry Men (1957)

Movies in both top-10 lists: 9


We compare my ten highest-scoring eligible movies from Model 2
with the baseline's top ten. Movies I have already rated are excluded.

The predictions shown are raw model scores, following the handout.
They are estimates rather than actual ratings and may fall outside
the 1–5 range.

The overlap counts how many movies appear in both lists.

In [36]:
# Isolate the adjustment added by SVD beyond the baseline.
personal = mine - base

print("Largest positive score adjustments:")
for movie in top(personal, n=6):
    print(f"  {personal[movie]:+.3f} | {titles[movie]}")

print("\nLargest negative score adjustments:")
for movie in top(-personal, n=6):
    print(f"  {personal[movie]:+.3f} | {titles[movie]}")

Largest positive score adjustments:
  +0.083 | Twelve Monkeys (1995)
  +0.078 | Dante's Peak (1997)
  +0.076 | Heathers (1989)
  +0.076 | Star Trek: First Contact (1996)
  +0.074 | Natural Born Killers (1994)
  +0.071 | 2001: A Space Odyssey (1968)

Largest negative score adjustments:
  -0.123 | Godfather, The (1972)
  -0.115 | Jerry Maguire (1996)
  -0.112 | It's a Wonderful Life (1946)
  -0.110 | Rock, The (1996)
  -0.105 | English Patient, The (1996)
  -0.099 | Raiders of the Lost Ark (1981)


Subtracting the baseline from Model 2's prediction isolates the
personal adjustment contributed by SVD.

Positive adjustments raise a movie's predicted score relative to
the baseline. Negative adjustments lower it. These are changes in
predicted score, not the number of positions moved in the ranking.

## Exercise 2: My Recommendations

Model 2 and the baseline share 9 of their top 10 recommendations.
Model 2 includes As Good As It Gets, while the baseline includes
Star Wars instead.

The personal adjustments are relatively small. Among eligible movies,
Twelve Monkeys receives the largest increase, approximately 0.083 stars,
and The Godfather receives the largest decrease, approximately
0.123 stars.

This shows that the baseline strongly influences the recommendations.
The SVD changes some scores and rankings, but it does not produce
a completely different list from only ten personal ratings.

## Exercise 3: The Opposite Person

We create an imaginary friend who rates the same movies but changes
every 5-star rating to 1 star and every 1-star rating to 5 stars.
All other ratings stay unchanged.

My ratings contain three 5-star ratings and no 1-star ratings.
Therefore, this experiment changes Home Alone, Titanic, and Jumanji
from 5 stars to 1 star. It does not reverse every preference.

In [37]:
# Swap only ratings of 5 and 1, as requested in the exercise.
friend_ratings = {}

for movie_id, stars in my_ratings.items():
    if stars == 5:
        friend_ratings[movie_id] = 1
    elif stars == 1:
        friend_ratings[movie_id] = 5
    else:
        friend_ratings[movie_id] = stars

# Add the friend to the original dataset instead of adding both of us.
friend = np.full((1, R.shape[1]), np.nan)

for movie_id, stars in friend_ratings.items():
    friend[0, movie_id - 1] = stars

R_friend = np.vstack([R, friend])

# Refit Model 2 using the friend's ratings.
known_friend = ~np.isnan(R_friend)
B_friend = baseline_matrix(R_friend)
X_friend = np.where(
    known_friend, R_friend - B_friend, 0.0
)

U_friend, s_friend, Vt_friend = np.linalg.svd(
    X_friend, full_matrices=False
)

P_friend = rank_k(
    U_friend, s_friend, Vt_friend, chosen_k
) + B_friend

friend_base = B_friend[-1]
friend_personal = P_friend[-1] - friend_base

# The same movies were rated, so the existing eligibility mask
# and top() function can be used for both people.
my_up = set(top(personal, 6))
my_down = set(top(-personal, 6))
friend_up = set(top(friend_personal, 6))
friend_down = set(top(-friend_personal, 6))

print("Friend: largest positive adjustments")
for movie in top(friend_personal, 6):
    print(f"  {friend_personal[movie]:+.3f} | {titles[movie]}")

print("\nFriend: largest negative adjustments")
for movie in top(-friend_personal, 6):
    print(f"  {friend_personal[movie]:+.3f} | {titles[movie]}")

# Identify movies appearing in opposite top-six adjustment lists.
print("\nMy moved-up list -> friend's moved-down list:")
print([titles[m] for m in sorted(my_up & friend_down)])

print("\nMy moved-down list -> friend's moved-up list:")
print([titles[m] for m in sorted(my_down & friend_up)])

friend_baseline_top10 = top(friend_base)

print("\nFriend's baseline top 10:")
for movie in friend_baseline_top10:
    print(f"  {friend_base[movie]:.2f} | {titles[movie]}")

print(
    "\nBaseline top-10 overlap:",
    len(set(baseline_top10) & set(friend_baseline_top10))
)

print(
    "Same baseline order:",
    np.array_equal(baseline_top10, friend_baseline_top10)
)

Friend: largest positive adjustments
  +0.101 | Liar Liar (1997)
  +0.085 | Terminator 2: Judgment Day (1991)
  +0.072 | Aliens (1986)
  +0.067 | Ace Ventura: Pet Detective (1994)
  +0.066 | Star Trek: The Wrath of Khan (1982)
  +0.060 | Star Trek IV: The Voyage Home (1986)

Friend: largest negative adjustments
  -0.224 | Fargo (1996)
  -0.178 | Air Force One (1997)
  -0.125 | English Patient, The (1996)
  -0.109 | Contact (1997)
  -0.102 | Rock, The (1996)
  -0.088 | Birdcage, The (1996)

My moved-up list -> friend's moved-down list:
[]

My moved-down list -> friend's moved-up list:
[]

Friend's baseline top 10:
  3.63 | Close Shave, A (1995)
  3.55 | Wrong Trousers, The (1993)
  3.53 | Schindler's List (1993)
  3.53 | Shawshank Redemption, The (1994)
  3.49 | Casablanca (1942)
  3.48 | Wallace & Gromit: The Best of Aardman Animation (1996)
  3.48 | Usual Suspects, The (1995)
  3.46 | Good Will Hunting (1997)
  3.45 | Star Wars (1977)
  3.41 | 12 Angry Men (1957)

Baseline top-10 over

We compare the six largest positive and negative adjustments for
both people. An empty swap list means no movie appears in both
of the specified top-six lists; it does not mean no scores changed.

For a given fitted baseline, the new user's mean shifts all movie
scores equally, so it does not personalize their ordering.
Refitting can also change movie effects slightly because the new
user's ratings contribute to those effects. Therefore, the baseline
rankings may remain similar without being guaranteed identical.

Based on my ratings, I would be less interested in Casablanca than in
a light comedy or adventure movie. I gave Home Alone and Jumanji
5 stars, so Casablanca seems less aligned with those preferences.
That does not mean it is a bad recommendation; I have not rated it.

Some personal adjustments are surprising. The increase for Dante's
Peak seems plausible given my interest in Jurassic Park and Jumanji.
However, lowering Raiders of the Lost Ark seems less consistent with
that interest in adventure movies.

Overall, the recommendations only partly reflect my preferences.
Nine of the ten recommendations also appear in the baseline list,
so the model still relies heavily on general movie ratings.

### Exercise 3 Results

No movies swapped between my top-six positive adjustments and my
friend's top-six negative adjustments, or in the reverse direction.
However, the adjustment lists did change. My friend's largest increase
was for Liar Liar (+0.101), and the largest decrease was for Fargo
(-0.224).

Only three ratings were changed, so the friend's preferences are
not a complete reversal of mine.

The baseline recommended the same ten movies in exactly the same
order. Its scores were about 1.2 stars lower for the friend.
Changing three ratings from 5 to 1 lowered the average of the ten
ratings by 12/10 = 1.2 stars.

For eligible movies that neither person rated, the movie effects
remain unchanged in this one-pass baseline. The lower user average
therefore shifts their baseline scores equally without changing
their order.

## Exercise 4: Does Rank Matter?

We compare my personal adjustments at k = 2, k = 15, and k = 50.
The original ratings and baseline stay the same.

We reuse the SVD of my expanded ratings matrix because changing
the retained rank does not require computing the SVD again.

In [38]:
# Compare the requested ranks with our validation-selected rank.
up_by_rank = {}

for k in [2, chosen_k, 50]:
    # Reconstruct predictions at this rank.
    P_k = rank_k(U_me, s_me, Vt_me, k) + B_me
    personal_k = P_k[-1] - B_me[-1]

    up_by_rank[k] = top(personal_k, 6)

    print(f"\nk = {k}: largest positive adjustments")
    for movie in up_by_rank[k]:
        print(f"  {personal_k[movie]:+.3f} | {titles[movie]}")

# Count shared movies in the top-six adjustment lists.
for k in [2, 50]:
    shared = (
        set(up_by_rank[k])
        & set(up_by_rank[chosen_k])
    )

    print(
        f"\nTop-six overlap between k = {k} "
        f"and k = {chosen_k}: {len(shared)}"
    )


k = 2: largest positive adjustments
  +0.012 | Brazil (1985)
  +0.012 | Clockwork Orange, A (1971)
  +0.012 | To Die For (1995)
  +0.011 | Remains of the Day, The (1993)
  +0.010 | Trainspotting (1996)
  +0.008 | Crumb (1994)

k = 15: largest positive adjustments
  +0.083 | Twelve Monkeys (1995)
  +0.078 | Dante's Peak (1997)
  +0.076 | Heathers (1989)
  +0.076 | Star Trek: First Contact (1996)
  +0.074 | Natural Born Killers (1994)
  +0.071 | 2001: A Space Odyssey (1968)

k = 50: largest positive adjustments
  +0.153 | Evita (1996)
  +0.139 | L.A. Confidential (1997)
  +0.131 | Leaving Las Vegas (1995)
  +0.125 | Die Hard (1988)
  +0.115 | Carrie (1976)
  +0.112 | Speed (1994)

Top-six overlap between k = 2 and k = 15: 0

Top-six overlap between k = 50 and k = 15: 0


A smaller rank keeps fewer broad patterns, while a larger rank
includes additional detail. This can change which movies receive
the largest positive adjustments.

In Milestone 3, increasing rank beyond the best setting increased
test error. Therefore, a more detailed recommendation pattern is
not automatically more accurate.

These lists show sensitivity to rank, but they cannot establish
accuracy because we do not know my actual ratings for these movies.

### Exercise 4 Results

Changing k changed all six movies in the largest-positive-adjustment
list when comparing k = 2 or k = 50 with k = 15.

At k = 2, the largest increase was about 0.012 stars for Brazil.
At k = 15, it was about 0.083 stars for Twelve Monkeys.
At k = 50, it was about 0.153 stars for Evita.

The larger ranks produced stronger maximum positive adjustments
in this example, but stronger adjustments do not prove greater accuracy.
In Milestone 3, Model 2's test RMSE increased from 0.9250 at k = 15
to 0.9363 at k = 50.

This supports keeping our validation-selected rank of 15 rather than
choosing a larger rank simply because its recommendations change more.

## Main Finding

Using ten of my movie ratings, the model raised Twelve Monkeys by
about 0.08 stars and lowered The Godfather by about 0.12 stars
relative to the baseline, but nine of its top ten recommendations
were still the same as the baseline's.